# QMatBridge Notebook Quickstart

This notebook gives a simple, visual workflow for users:
1. Generate ranking and Pareto artifacts from the example scripts
2. Load CSV outputs with pandas
3. Plot and compare candidates directly in the notebook

In [ ]:
from pathlib import Path
import subprocess
import sys

import matplotlib.pyplot as plt
import pandas as pd

repo_root = Path.cwd()
examples_dir = repo_root / "examples"
print(f"Repo root: {repo_root}")

## Ensure input artifacts exist

If the CSV files are missing, this cell runs the example scripts to generate them.

In [ ]:
needed = [
    examples_dir / "material_ranking.csv",
    examples_dir / "pareto_candidates.csv",
]

if not all(path.exists() for path in needed):
    print("Generating artifacts from scripts...")
    subprocess.run([sys.executable, "examples/material_selection_visual.py"], check=True)
    subprocess.run([sys.executable, "examples/pareto_frontier.py"], check=True)
else:
    print("All required CSV files already exist.")

for path in needed:
    print(f"- {path} ({path.stat().st_size} bytes)")

## Load ranking results

In [ ]:
ranking = pd.read_csv(examples_dir / "material_ranking.csv")
pareto = pd.read_csv(examples_dir / "pareto_candidates.csv")

display(ranking)
display(pareto)

## Visual 1: Composite score ranking

In [ ]:
plot_df = ranking.copy()
plot_df["label"] = plot_df["formula"] + " (" + plot_df["source_id"] + ")"
plot_df = plot_df.sort_values("score", ascending=True)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.barh(plot_df["label"], plot_df["score"], color=["#2a9d8f", "#4c78a8", "#e76f51"])
ax.invert_yaxis()
ax.set_title("Material ranking by composite score")
ax.set_xlabel("Score (lower is better)")
ax.grid(axis="x", linestyle="--", alpha=0.35)

for y, value in enumerate(plot_df["score"]):
    ax.text(value * 1.01, y, f"{value:,.0f}", va="center", fontsize=9)

fig.tight_layout()
plt.show()

## Visual 2: Pareto frontier view

In [ ]:
pareto["is_pareto_optimal"] = pareto["is_pareto_optimal"].astype(bool)
dom = pareto[~pareto["is_pareto_optimal"]]
front = pareto[pareto["is_pareto_optimal"]].sort_values("qubits_proxy")

fig, ax = plt.subplots(figsize=(8, 5))
if not dom.empty:
    ax.scatter(dom["qubits_proxy"], dom["t_gate_proxy"], s=65, color="#9aa0a6", label="Dominated")

ax.scatter(front["qubits_proxy"], front["t_gate_proxy"], s=85, color="#2a9d8f", label="Pareto-optimal", edgecolors="#1b4332")
if len(front) > 1:
    ax.plot(front["qubits_proxy"], front["t_gate_proxy"], linestyle="--", color="#2a9d8f")

for _, row in pareto.iterrows():
    ax.annotate(row["formula"], (row["qubits_proxy"], row["t_gate_proxy"]), xytext=(5, 5), textcoords="offset points", fontsize=8)

ax.set_title("Pareto frontier: qubits vs T-gate proxy")
ax.set_xlabel("Qubit proxy (lower is better)")
ax.set_ylabel("T-gate proxy (lower is better)")
ax.grid(True, linestyle="--", alpha=0.3)
ax.legend(frameon=False)
fig.tight_layout()
plt.show()

## Decision summary

In [ ]:
best = ranking.sort_values("score").iloc[0]
front_list = pareto.loc[pareto["is_pareto_optimal"], ["formula", "source_id"]]

print("Recommended first benchmark candidate:")
print(f"- {best['formula']} ({best['source_id']}) with score {best['score']:,.0f}")
print("\nPareto-optimal shortlist:")
for _, row in front_list.iterrows():
    print(f"- {row['formula']} ({row['source_id']})")